In [1]:
import pandas as pd

In [2]:
author_df = pd.read_csv("../data/GSE117588_organoids-gene-counts.tsv.gz", sep="\t")
NCBI_df = pd.read_csv("../data/GSE117588_raw_counts_GRCh38.p13_NCBI.tsv.gz", sep="\t")
gene_table_df = pd.read_csv("../data/Human.GRCh38.p13.annot.tsv.gz", sep="\t", dtype={"ChrStart": str, "ChrStop": str})

In [24]:
def inspect_files(author_df):
    print(f"File shape: {author_df.shape}")
    
    #Checking suffixes in gene ID column of file
    has_versions = author_df['geneId'].str.contains(r'\.\d+$', na=False).any()
    if has_versions:
        print("Gene IDs in author's count contain suffix with up to 2 places after decimals")
    else:
        print("Gene IDs in author's count do not contain any suffix")

    #Checking if numerical column contains whole number only. True indicates all whole numbers
    num_df = author_df.select_dtypes(include='number')
    is_whole_per_col = ((num_df % 1 == 0) & (num_df >= 0)).all()
    print(is_whole_per_col)

    #Checking library sizes
    lib_sizes_m = (num_df.sum() / 1e6).round(2)
    print("Library sizes (millions):")
    print(lib_sizes_m)

    # Number of expressed genes (count > 0) per sample
    detected_genes = (num_df > 0).sum()
    print("\nDetected genes per sample:")
    print(detected_genes)

    #Number of genes having zero count in all 6 samples
    all_zero_genes = (num_df == 0).all(axis=1).sum()
    print(f"Genes with 0 counts in all 6 samples: {all_zero_genes}")
    

    #Updating metadata
    existing = pd.read_csv('../metadata/samples.csv')
    if 'sample' not in existing.columns:
        existing = existing.iloc[:, 0].str.split(',', expand=True)
        existing.columns = ['run', 'sample', 'group', 'replicate']
        existing = existing['run,sample,group,replicate'].str.split(',', expand=True)
        existing.columns = ['run', 'sample', 'group', 'replicate']
    existing = existing.sort_values(by='sample', ascending=True).reset_index(drop=True)
    existing['count_column'] = list(num_df.columns)
    existing.to_csv('../metadata/samples.csv', index=False)
    print(existing)
    


In [25]:
inspect_files(author_df)

File shape: (58395, 7)
Gene IDs in author's count do not contain any suffix
ADR_11    True
ADR_12    True
ADR_13    True
ADR_14    True
ADR_5     True
ADR_6     True
dtype: bool
Library sizes (millions):
ADR_11    10.54
ADR_12     9.08
ADR_13    12.05
ADR_14    12.97
ADR_5     10.31
ADR_6     11.80
dtype: float64

Detected genes per sample:
ADR_11    26670
ADR_12    26713
ADR_13    26763
ADR_14    26837
ADR_5     27272
ADR_6     26280
dtype: int64
Genes with 0 counts in all 6 samples: 23791
          run      sample     group  replicate count_column
0  SRR7586681  GSM3304324  E4E4_CTR          1       ADR_11
1  SRR7586682  GSM3304325  E4E4_CTR          2       ADR_12
2  SRR7586683  GSM3304326  E3E3_ISO          1       ADR_13
3  SRR7586684  GSM3304327  E3E3_ISO          2       ADR_14
4  SRR7586685  GSM3304328  E4E4_CTR          3        ADR_5
5  SRR7586686  GSM3304329  E3E3_ISO          3        ADR_6
